# Run Allocations

Generated from `src/run_allocations.py`.
The first code cell recreates script-like path behavior for notebook execution.


In [ ]:
from pathlib import Path
import sys

_NOTEBOOK_BOOTSTRAP_VERBOSE = True

if _NOTEBOOK_BOOTSTRAP_VERBOSE:
    print("[bootstrap] cwd =", Path.cwd().resolve())

if "ipykernel" in sys.modules:
    # Avoid argparse failures from Jupyter kernel launch flags.
    sys.argv = [sys.argv[0]]
    if _NOTEBOOK_BOOTSTRAP_VERBOSE:
        print("[bootstrap] detected ipykernel, trimmed sys.argv to:", sys.argv)

_SOURCE_RELATIVE_PATH = Path("src/run_allocations.py")
_repo_root = None
for _candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if _NOTEBOOK_BOOTSTRAP_VERBOSE:
        print("[bootstrap] checking candidate:", _candidate)
    if (_candidate / _SOURCE_RELATIVE_PATH).exists():
        _repo_root = _candidate
        if _NOTEBOOK_BOOTSTRAP_VERBOSE:
            print("[bootstrap] matched repo root:", _repo_root)
        break

if _repo_root is None:
    _repo_root = Path.cwd().resolve()
    if _NOTEBOOK_BOOTSTRAP_VERBOSE:
        print("[bootstrap] no match found, falling back to cwd:", _repo_root)

_source_file = (_repo_root / _SOURCE_RELATIVE_PATH).resolve()
__file__ = str(_source_file)
if _NOTEBOOK_BOOTSTRAP_VERBOSE:
    print("[bootstrap] source relative path =", _SOURCE_RELATIVE_PATH)
    print("[bootstrap] resolved __file__ =", __file__)

for _path in (str(_repo_root), str(_source_file.parent)):
    if _path not in sys.path:
        sys.path.insert(0, _path)
        if _NOTEBOOK_BOOTSTRAP_VERBOSE:
            print("[bootstrap] added to sys.path:", _path)
    elif _NOTEBOOK_BOOTSTRAP_VERBOSE:
        print("[bootstrap] already on sys.path:", _path)


In [ ]:
from __future__ import annotations

import argparse
import json
from typing import Dict, List

import sys
from pathlib import Path
sys.path.insert(0, str(Path(__file__).parent.parent))

try:
    from .model_train import MODEL_REGISTRY, run_single_experiment
except ImportError:
    from src.model_train import MODEL_REGISTRY, run_single_experiment


MODEL_ALLOCATIONS: Dict[str, List[str]] = {
    "ethics": [
        "tfidf_logreg",
        "tfidf_linearsvc",
        "distilbert-base-uncased",
        "bert-base-uncased",
        "roberta-base",
        "microsoft/deberta-v3-base",
    ],
    "normbank": [
        "tfidf_logreg",
        "bow_mnb",
        "distilbert-base-uncased",
        "bert-base-uncased",
        "roberta-base",
    ],
    "mfrc": [
        "tfidf_logreg",
        "distilbert-base-uncased",
        "bert-base-uncased",
    ],
    "moralbench": [
        "t5-small",
        "facebook/bart-base",
    ],
    "delphi": [
        "tfidf_logreg",
        "distilbert-base-uncased",
        "bert-base-uncased",
        "t5-small",
    ],
}


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Run the model-to-dataset allocations from the logbook.")
    parser.add_argument("--datasets", nargs="*", default=[], help="Subset of datasets to run.")
    parser.add_argument("--max-train-samples", type=int, default=None)
    parser.add_argument("--max-test-samples", type=int, default=None)
    parser.add_argument("--epochs", type=int, default=1)
    return parser.parse_args()


def main() -> None:
    args = parse_args()
    targets = args.datasets if args.datasets else list(MODEL_ALLOCATIONS.keys())
    outputs = []

    for dataset in targets:
        for model in MODEL_ALLOCATIONS.get(dataset, []):
            if model not in MODEL_REGISTRY:
                continue
            exp_args = argparse.Namespace(
                dataset=dataset,
                model=model,
                max_train_samples=args.max_train_samples,
                max_test_samples=args.max_test_samples,
                epochs=args.epochs,
            )
            try:
                row = run_single_experiment(exp_args)
                row["status"] = "ok"
            except Exception as exc:
                row = {
                    "dataset": dataset,
                    "model": model,
                    "status": "skipped",
                    "reason": str(exc),
                }
            outputs.append(row)
            print(json.dumps(row, ensure_ascii=False))

    print(json.dumps({"total": len(outputs)}, ensure_ascii=False))


## Entrypoint

Run the original script entrypoint when needed.


In [ ]:
main()
